In [23]:
import os
os.environ["MKL_NUM_THREADS"] = "5"
os.environ["OPENBLAS_NUM_THREADS"] = "5"
os.environ["NUMEXPR_NUM_THREADS"] = "5"

# Example GLM standard framework

In [25]:
from importlib.resources import files
import pandas as pd
import numpy as np
from lepto.gui.framework.glm_framework import create_offset, create_graph_matrix, create_categories, GLMFramework
from lepto.gui.framework.data_preprocessor import DataPreprocessor

## Data

In [26]:
path = files("lepto.data") / "sample_standard.csv"
df = pd.read_csv(path)

In [27]:
X = df[["age", "region", "segment"]]
y = df["y"].values
w = df["exposure"].values

In [28]:
# Create fake data lon and lat for example
data_lat_long = pd.DataFrame({
    "region": ["north", "south", "east", "west"],
    "lat": [48.8566, 45.7640, 43.2965, 50.6292],
    "lon": [2.3522, 4.8357, 5.3698, 3.0573]
})

## Framework

### Init

In [29]:
# Launch Data preparation
var_for_model = ["age", "region", "segment"]
preproc = DataPreprocessor(
            df=df,
            variables=var_for_model,
            sparse_output=True)
X= preproc.run()

In [30]:
# Create default values
transformer_data = preproc.transformer_data
## Variables types and categories
geographical_data_dict = {}
geographical_data_dict["region"] = data_lat_long
variable_types = {"age": "continuous", "region": "geographical", "segment": "categorical"}
categories_list = create_categories(transformer_data, variable_types, geographical_data_dict)
categories_dict = dict(zip(transformer_data.var_num + transformer_data.var_cat, categories_list))
## Default offset
default_offset = create_offset(transformer_data, categories_list)
## Default adjacendy matrix
categories_var = transformer_data._get_categories_var()
adj = create_graph_matrix(variable_types, categories_var, geographical_data_dict)


### Launch

In [31]:
glm = GLMFramework(family="poisson",
                        tweedie_power = 1.5,
                        lam_grid=np.linspace(1e-6, 10, 10),
                        adj_matrix=adj,
                        offset_betas=default_offset,
                        categories=categories_list,
                        sparse_output=True)
glm.fit(X, y, w)

In [32]:
best_glm = glm.grid_search.best_estimator_

In [33]:
best_glm.plot(X, y, w, "region")